In [1]:
from dataclasses import dataclass


CAPACITY_A = 8
CAPACITY_B = 5
CAPACITY_C = 3
TOTAL_WATER = 8


@dataclass
class Node:
    state: tuple[int, int, int]
    parent: "Node | None"
    action: str | None
    cost: int


def goal_test(state: tuple[int, int, int], target: int) -> bool:
    a, b, c = state
    return a == target or b == target or c == target


def move_gen(state: tuple[int, int, int]) -> list[tuple[str, tuple[int, int, int]]]:
    a, b, c = state
    moves = []

    candidates = []

    if a < CAPACITY_A:
        candidates.append((f"Fill {CAPACITY_A}L jug", (CAPACITY_A, b, c)))

    candidates.append((f"Empty {CAPACITY_B}L jug", (a, 0, c)))
    candidates.append((f"Empty {CAPACITY_C}L jug", (a, b, 0)))

    pour = min(a, CAPACITY_B - b)
    candidates.append((f"Pour {CAPACITY_A}L jug into {CAPACITY_B}L jug", (a - pour, b + pour, c)))

    pour = min(a, CAPACITY_C - c)
    candidates.append((f"Pour {CAPACITY_A}L jug into {CAPACITY_C}L jug", (a - pour, b, c + pour)))

    pour = min(b, CAPACITY_A - a)
    candidates.append((f"Pour {CAPACITY_B}L jug into {CAPACITY_A}L jug", (a + pour, b - pour, c)))

    pour = min(b, CAPACITY_C - c)
    candidates.append((f"Pour {CAPACITY_B}L jug into {CAPACITY_C}L jug", (a, b - pour, c + pour)))

    pour = min(c, CAPACITY_A - a)
    candidates.append((f"Pour {CAPACITY_C}L jug into {CAPACITY_A}L jug", (a + pour, b, c - pour)))

    pour = min(c, CAPACITY_B - b)
    candidates.append((f"Pour {CAPACITY_C}L jug into {CAPACITY_B}L jug", (a, b + pour, c - pour)))

    seen = set()
    for action, next_state in candidates:
        total = sum(next_state)
        if next_state != state and next_state not in seen and total <= TOTAL_WATER:
            seen.add(next_state)
            moves.append((action, next_state))

    return moves


if __name__ == "__main__":
    print("=== Three Water Jug Problem (8L, 5L, 3L) ===\n")

    target = int(input("Enter target volume to measure: "))

    initial_a = int(input("Enter initial water in 8L jug (0-8): "))
    initial_b = int(input("Enter initial water in 5L jug (0-5): "))
    initial_c = int(input("Enter initial water in 3L jug (0-3): "))
    initial_state = (initial_a, initial_b, initial_c)

    print(f"\n--- Testing GoalTest ---")
    print(f"State: {initial_state}, Target: {target} -> GoalTest: {goal_test(initial_state, target)}")

    print(f"\n--- Testing MoveGen ---")
    print(f"State: {initial_state}")
    print(f"\nPossible moves:")
    for action, next_state in move_gen(initial_state):
        print(f"  {action} -> {next_state} (total: {sum(next_state)}L)")


=== Three Water Jug Problem (8L, 5L, 3L) ===


--- Testing GoalTest ---
State: (8, 0, 0), Target: 4 -> GoalTest: False

--- Testing MoveGen ---
State: (8, 0, 0)

Possible moves:
  Pour 8L jug into 5L jug -> (3, 5, 0) (total: 8L)
  Pour 8L jug into 3L jug -> (5, 0, 3) (total: 8L)


In [5]:
from dataclasses import dataclass


DISTANCE_MATRIX = {
    'A': {'A': 0, 'B': 10, 'C': 15},
    'B': {'A': 10, 'B': 0, 'C': 35},
    'C': {'A': 15, 'B': 35, 'C': 0},
}
CITIES = {'A', 'B', 'C'}


@dataclass
class Node:
    state: tuple[str, frozenset, int]
    parent: "Node | None"
    action: str | None
    cost: int


def goal_test(state: tuple[str, frozenset, int], all_cities: set, start_city: str) -> bool:
    current, visited, _ = state
    return visited == all_cities and current == start_city


def move_gen(
    state: tuple[str, frozenset, int], distance_matrix: dict, all_cities: set, start_city: str
) -> list[tuple[str, tuple[str, frozenset, int]]]:
    current, visited, dist = state
    moves = []

    if visited == all_cities:
        return_dist = distance_matrix[current][start_city]
        new_state = (start_city, visited, dist + return_dist)
        moves.append((f"Return to {start_city}", new_state))
    else:
        for city in all_cities:
            if city not in visited:
                travel_dist = distance_matrix[current][city]
                new_visited = visited | {city}
                new_state = (city, new_visited, dist + travel_dist)
                moves.append((f"Go to {city}", new_state))

    return moves


if __name__ == "__main__":
    print("=== Travelling Salesman Problem (3 Cities) ===\n")

    start_city = input("Enter start city (A, B, or C): ").strip().upper()
    if start_city not in CITIES:
        start_city = 'A'
        print(f"Invalid input. Using default: {start_city}")

    initial_state = (start_city, frozenset({start_city}), 0)

    print(f"\n--- Testing GoalTest ---")
    print(f"State: {initial_state}, GoalTest: {goal_test(initial_state, CITIES, start_city)}")

    print(f"\n--- Testing MoveGen ---")
    print(f"State: {initial_state}")
    print(f"\nPossible moves:")
    for action, next_state in move_gen(initial_state, DISTANCE_MATRIX, CITIES, start_city):
        current, visited, total_dist = next_state
        print(f"  {action} -> ({current}, {set(visited)}, {total_dist})")


=== Travelling Salesman Problem (3 Cities) ===


--- Testing GoalTest ---
State: ('B', frozenset({'B'}), 0), GoalTest: False

--- Testing MoveGen ---
State: ('B', frozenset({'B'}), 0)

Possible moves:
  Go to A -> (A, {'B', 'A'}, 10)
  Go to C -> (C, {'B', 'C'}, 35)


In [6]:
from dataclasses import dataclass


GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


@dataclass
class Node:
    state: tuple[int, ...]
    parent: "Node | None"
    action: str | None
    cost: int


def find_blank(state: tuple[int, ...]) -> int:
    return state.index(0)


def swap(state: tuple[int, ...], i: int, j: int) -> tuple[int, ...]:
    lst = list(state)
    lst[i], lst[j] = lst[j], lst[i]
    return tuple(lst)


def goal_test(state: tuple[int, ...]) -> bool:
    return state == GOAL


def move_gen(state: tuple[int, ...]) -> list[tuple[str, tuple[int, ...]]]:
    blank_idx = find_blank(state)
    row, col = blank_idx // 3, blank_idx % 3
    moves = []

    if row > 0:
        new_state = swap(state, blank_idx, blank_idx - 3)
        moves.append(("Up", new_state))

    if row < 2:
        new_state = swap(state, blank_idx, blank_idx + 3)
        moves.append(("Down", new_state))

    if col > 0:
        new_state = swap(state, blank_idx, blank_idx - 1)
        moves.append(("Left", new_state))

    if col < 2:
        new_state = swap(state, blank_idx, blank_idx + 1)
        moves.append(("Right", new_state))

    return moves


def print_state(state: tuple[int, ...]) -> None:
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else "_" for x in row))


if __name__ == "__main__":
    print("=== 8 Puzzle Problem ===\n")

    print("Enter initial state as 9 space-separated integers (0 for blank):")
    print("(default: 1 2 3 4 0 6 7 5 8)")
    user_input = input().strip()

    try:
        if user_input:
            initial_state = tuple(map(int, user_input.split()))
            if len(initial_state) != 9:
                raise ValueError
        else:
            initial_state = (1, 2, 3, 4, 0, 6, 7, 5, 8)
    except ValueError:
        initial_state = (1, 2, 3, 4, 0, 6, 7, 5, 8)
        print("Invalid input. Using default state.")

    print(f"\n--- Initial State ---")
    print_state(initial_state)

    print(f"\n--- Testing GoalTest ---")
    print(f"GoalTest: {goal_test(initial_state)}")

    print(f"\n--- Testing MoveGen ---")
    print(f"Possible moves:")
    for action, next_state in move_gen(initial_state):
        print(f"\n  {action}:")
        print_state(next_state)


=== 8 Puzzle Problem ===

Enter initial state as 9 space-separated integers (0 for blank):
(default: 1 2 3 4 0 6 7 5 8)

--- Initial State ---
1 2 _
3 4 5
6 7 8

--- Testing GoalTest ---
GoalTest: False

--- Testing MoveGen ---
Possible moves:

  Down:
1 2 5
3 4 _
6 7 8

  Left:
1 _ 2
3 4 5
6 7 8
